In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Labs"
)

xpt_file = DATA_DIR / "HDL_H.xpt"
csv_file = DATA_DIR / "HDL_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the completely untouched source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 HDL_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBDHDD",
    "LBDHDDSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HDL_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 8291:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 8,291 rows x 3 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# LBDHDDSI is a genuine decimal measurement.
#
# DO NOT:
#
#     round it
#     convert it to Int64
#     globally remove ".0"
#     use float_format during export
#
# LBDHDD is released as whole-number mg/dL values.
#

DECIMAL_COLUMNS = [
    "LBDHDDSI"
]


INTEGER_COLUMNS = [
    "SEQN",
    "LBDHDD"
]


if len(DECIMAL_COLUMNS) != 1:

    raise ValueError(
        "Expected exactly one genuine decimal field."
    )


if len(INTEGER_COLUMNS) != 2:

    raise ValueError(
        "Expected exactly two whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT artifact:
#
#     5.397605346934028e-79
#
# Attached HDL_H contains none.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Tiny XPORT artifacts:",
    tiny_total
)

print(
    "Ordinary numeric zeros:",
    true_zero_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HDL_H."
    )


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected zero values found "
        "in attached HDL_H."
    )


print(
    "PASS: No tiny artifacts or ordinary zeros."
)


# ============================================================
# 6. DETECT GENUINE DECIMAL FIELDS
# ============================================================

#
# Run BEFORE converting whole-number fields.
#
# np.round() is used ONLY for testing whether
# values are fractional.
#
# It does NOT modify the data.
#

print("\n--- GENUINE DECIMAL DETECTION ---")


detected_decimal_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-field pattern differs.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {detected_decimal_columns}"
    )


print(
    "PASS: LBDHDDSI is the only "
    "genuine decimal field."
)


# ============================================================
# 7. VALIDATE LBDHDDSI DECIMAL FIELD
# ============================================================

s = df["LBDHDDSI"]


decimal_nonmissing = int(
    s.notna()
    .sum()
)


decimal_missing = int(
    s.isna()
    .sum()
)


decimal_min = float(
    s.dropna()
    .min()
)


decimal_max = float(
    s.dropna()
    .max()
)


print("\n--- LBDHDDSI ---")

print(
    "Nonmissing:",
    decimal_nonmissing
)

print(
    "Missing:",
    decimal_missing
)

print(
    "Minimum:",
    decimal_min
)

print(
    "Maximum:",
    decimal_max
)


if decimal_nonmissing != 7624:

    raise ValueError(
        "Unexpected LBDHDDSI nonmissing count."
    )


if decimal_missing != 667:

    raise ValueError(
        "Unexpected LBDHDDSI missing count."
    )


if not np.isclose(
    decimal_min,
    0.26,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDHDDSI minimum."
    )


if not np.isclose(
    decimal_max,
    4.47,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDHDDSI maximum."
    )


print(
    "PASS: LBDHDDSI range/count validated."
)


# ============================================================
# 8. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 9. VALIDATE LBDHDD AS WHOLE-NUMBER FIELD
# ============================================================

hdd_values = (
    df["LBDHDD"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    hdd_values,
    np.round(hdd_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "LBDHDD unexpectedly contains "
        "fractional values."
    )


df["LBDHDD"] = (
    df["LBDHDD"]
    .astype("Int64")
)


print(
    "PASS: LBDHDD safely converted "
    "to nullable Int64."
)


# ============================================================
# 10. CDC CHECK - LBDHDD
# ============================================================

#
# Direct HDL cholesterol in mg/dL.
#
# CDC:
#
# range      = 10-173
# nonmissing = 7,624
# missing    = 667
#

s = df["LBDHDD"]


hdd_nonmissing = int(
    s.notna()
    .sum()
)


hdd_missing = int(
    s.isna()
    .sum()
)


hdd_min = int(
    s.dropna()
    .min()
)


hdd_max = int(
    s.dropna()
    .max()
)


print("\n--- LBDHDD ---")

print(
    "Nonmissing:",
    hdd_nonmissing
)

print(
    "Missing:",
    hdd_missing
)

print(
    "Minimum:",
    hdd_min
)

print(
    "Maximum:",
    hdd_max
)


if hdd_nonmissing != 7624:

    raise ValueError(
        "Unexpected LBDHDD nonmissing count."
    )


if hdd_missing != 667:

    raise ValueError(
        "Unexpected LBDHDD missing count."
    )


if hdd_min != 10:

    raise ValueError(
        "Unexpected LBDHDD minimum."
    )


if hdd_max != 173:

    raise ValueError(
        "Unexpected LBDHDD maximum."
    )


print(
    "PASS: LBDHDD matches CDC range/count."
)


# ============================================================
# 11. LBDHDD / LBDHDDSI MISSING-PATTERN CHECK
# ============================================================

#
# The mg/dL and mmol/L values should have exactly
# the same missing positions.
#

hdd_nonmissing_mask = (
    df["LBDHDD"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


si_nonmissing_mask = (
    df["LBDHDDSI"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


missing_pattern_mismatch = int(
    np.sum(
        hdd_nonmissing_mask
        !=
        si_nonmissing_mask
    )
)


print(
    "\n--- HDL MISSING-PATTERN CHECK ---"
)

print(
    "Mismatches:",
    missing_pattern_mismatch
)


if missing_pattern_mismatch != 0:

    raise ValueError(
        "LBDHDD and LBDHDDSI "
        "missing patterns differ."
    )


print(
    "PASS: HDL mg/dL and mmol/L "
    "missing patterns match exactly."
)


# ============================================================
# 12. CDC SI-CONVERSION CONSISTENCY CHECK
# ============================================================

#
# CDC documentation:
#
# HDL mg/dL -> mmol/L:
#
#     mg/dL * 0.02586
#
# LBDHDDSI is released to two decimal places.
#
# IMPORTANT:
#
# This check DOES NOT overwrite LBDHDDSI.
#
# We validate the released value only.
#

paired = (
    df[
        [
            "LBDHDD",
            "LBDHDDSI"
        ]
    ]
    .dropna()
)


calculated_si = np.round(
    paired["LBDHDD"]
    .to_numpy(
        dtype=float
    )
    *
    0.02586,
    2
)


released_si = (
    paired["LBDHDDSI"]
    .to_numpy(
        dtype=float
    )
)


si_mismatch_count = int(
    np.sum(
        calculated_si
        !=
        released_si
    )
)


print(
    "\n--- HDL SI CONVERSION CHECK ---"
)

print(
    "Conversion mismatches:",
    si_mismatch_count
)


if si_mismatch_count != 0:

    raise ValueError(
        "LBDHDDSI does not match the released "
        "mg/dL -> mmol/L conversion."
    )


print(
    "PASS: All 7,624 LBDHDDSI values match "
    "LBDHDD x 0.02586 rounded to 2 decimals."
)


# ============================================================
# 13. CONFIRM NO ZERO VALUES CREATED
# ============================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zeros were created:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 14. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# HDL_H requires NO numerical correction.
#
# Therefore all values must remain numerically
# identical to the source XPT.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 3 variables preserve "
    "source XPT numeric values exactly."
)


# ============================================================
# 15. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
# LBDHDDSI must remain its genuine
# decimal numeric value.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 16. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 17. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 3 CSV variables round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 18. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 19. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 20. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 21. .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# DO NOT perform this check globally.
#
# LBDHDDSI is a genuine decimal field.
#
# Only:
#
#     SEQN
#     LBDHDD
#
# must have unnecessary ".0" removed.
#

print(
    "\n--- INTEGER COLUMN .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "SEQN or LBDHDD."
)


# ============================================================
# 22. EXACT LBDHDDSI DECIMAL FIDELITY
# ============================================================

expected_si = (
    source_df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    )
)


exported_si = (
    roundtrip_df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    )
)


if not np.allclose(
    expected_si,
    exported_si,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBDHDDSI decimal fidelity failed."
    )


print(
    "PASS: LBDHDDSI decimal values "
    "preserved exactly."
)


# ============================================================
# 23. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zeros found in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 24. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 25. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83731:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "HDL_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Laboratory - HDL Cholesterol"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "LBDHDD nonmissing:",
    hdd_nonmissing
)


print(
    "LBDHDD range:",
    f"{hdd_min}-{hdd_max} mg/dL"
)


print(
    "LBDHDDSI nonmissing:",
    decimal_nonmissing
)


print(
    "LBDHDDSI range:",
    f"{decimal_min}-{decimal_max} mmol/L"
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number variables:",
    INTEGER_COLUMNS
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "LBDHDDSI decimal differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0' in integer fields: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 HDL_H XPT -> CSV
Rows: 8,291
Columns: 3
PASS: Structure = 8,291 rows x 3 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['LBDHDDSI']
Whole-number fields: ['SEQN', 'LBDHDD']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: No tiny artifacts or ordinary zeros.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
LBDHDD: 0 fractional observations
LBDHDDSI: 7,624 fractional observations
PASS: LBDHDDSI is the only genuine decimal field.

--- LBDHDDSI ---
Nonmissing: 7624
Missing: 667
Minimum: 0.26
Maximum: 4.47
PASS: LBDHDDSI range/count validated.

--- SEQN ---
Minimum: 73557
Maximum: 83731
Missing: 0
Duplicates: 0
PASS: SEQN 73557-83731 validated.
PASS: LBDHDD safely converted to nullable Int64.

--- LBDHDD ---
Nonmissing: 7624
Missing: 667
Minimum: 10
Maximum: 173
PASS: LBDHDD matches CDC range/count.

--- HDL MISSING-PATTERN CHECK ---
Mismatches: 0
PASS: HDL mg/dL and mmol/L missing patterns match exactly.